In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_recarga")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8041a4fc-d190-4f4f-b483-1b7da4d52f3b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 117ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/book_recarga/"
df_book_recarga = spark.read.parquet(path)
df_book_recarga.show(5, truncate=False)

25/12/31 09:10:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+
|NUM_CPF    |DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|
+-----------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+--

In [4]:
df_book_recarga.createOrReplaceTempView("raw_00")

In [5]:
df_book_recarga.count()

100213651

In [6]:

raw_00_com_safra = spark.sql("""
    SELECT
        *,
        CAST(
            date_format(
                to_timestamp(DAT_INSERCAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),
                'yyyyMM'
            ) AS INT
        ) AS SAFRA
    FROM raw_00
""")

raw_00_com_safra.createOrReplaceTempView("raw_00_com_safra")

In [7]:
raw_00_com_safra.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- DW_NUM_NTC: string (nullable = true)
 |-- DAT_INSERCAO_CREDITO: string (nullable = true)
 |-- HOR_INSERCAO_CREDITO: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- COD_TECNOLOGIA_DW: string (nullable = true)
 |-- COD_CANAL_AQUISICAO: string (nullable = true)
 |-- COD_TIPO_CREDITO: string (nullable = true)
 |-- COD_PROMOCAO: string (nullable = true)
 |-- VAL_CREDITO_INSERIDO: string (nullable = true)
 |-- VAL_BONUS: string (nullable = true)
 |-- VAL_REAL: string (nullable = true)
 |-- COD_PLATAFORMA_ATU: string (nullable = true)
 |-- COD_STATUS_PLATAFORMA: string (nullable = true)
 |-- IND_METODO_PAGAMENTO: string (nullable = true)
 |-- DW_PLANO_TARIFACAO: string (nullable = true)
 |-- DW_TIPO_RECARGA: string (nullable = true)
 |-- DW_TIPO_INSERCAO: string (nullable = true)
 |-- DW_FORMA_PAGAMENTO: string (nullable = true)
 |-- DW_INSTITUICAO: string (nullable = true)
 |-- COD_GRUPO_CARTAO: string (nullable = t

In [8]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT DW_NUM_NTC) as num_tel_distintos
    FROM raw_00_com_safra
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(truncate=False)

+------+------------+-------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|num_tel_distintos|
+------+------------+-------------+-----------------+
|202310|4366091     |1429755      |1759695          |
|202311|4312612     |1448184      |1783264          |
|202312|4609579     |1511629      |1875533          |
|202401|4397192     |1508640      |1866742          |
|202402|4444044     |1513604      |1879546          |
|202403|4981887     |1580382      |1982465          |
|202404|4890093     |1599384      |2011623          |
|202405|5102046     |1650460      |2090755          |
|202406|5150385     |1708324      |2172745          |
|202407|5549310     |1780332      |2292069          |
|202408|5903693     |1821869      |2364586          |
|202409|5849901     |1838158      |2378768          |
|202410|6849716     |2025433      |2660785          |
|202411|7187244     |2199993      |2947022          |
|202412|7321919     |2306817      |3117906          |
|202501|6691129     |2297663

In [9]:
print('lista de colunas para tipar')
for col in spark.table("raw_00_com_safra").columns:
    print('cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
cast(NUM_CPF as) as NUM_CPF,
cast(DW_NUM_NTC as) as DW_NUM_NTC,
cast(DAT_INSERCAO_CREDITO as) as DAT_INSERCAO_CREDITO,
cast(HOR_INSERCAO_CREDITO as) as HOR_INSERCAO_CREDITO,
cast(DW_NUM_CLIENTE as) as DW_NUM_CLIENTE,
cast(COD_TECNOLOGIA_DW as) as COD_TECNOLOGIA_DW,
cast(COD_CANAL_AQUISICAO as) as COD_CANAL_AQUISICAO,
cast(COD_TIPO_CREDITO as) as COD_TIPO_CREDITO,
cast(COD_PROMOCAO as) as COD_PROMOCAO,
cast(VAL_CREDITO_INSERIDO as) as VAL_CREDITO_INSERIDO,
cast(VAL_BONUS as) as VAL_BONUS,
cast(VAL_REAL as) as VAL_REAL,
cast(COD_PLATAFORMA_ATU as) as COD_PLATAFORMA_ATU,
cast(COD_STATUS_PLATAFORMA as) as COD_STATUS_PLATAFORMA,
cast(IND_METODO_PAGAMENTO as) as IND_METODO_PAGAMENTO,
cast(DW_PLANO_TARIFACAO as) as DW_PLANO_TARIFACAO,
cast(DW_TIPO_RECARGA as) as DW_TIPO_RECARGA,
cast(DW_TIPO_INSERCAO as) as DW_TIPO_INSERCAO,
cast(DW_FORMA_PAGAMENTO as) as DW_FORMA_PAGAMENTO,
cast(DW_INSTITUICAO as) as DW_INSTITUICAO,
cast(COD_GRUPO_CARTAO as) as COD_GRUPO_CARTAO,
c

In [12]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            try_cast(DW_NUM_NTC as BIGINT) as DW_NUM_NTC,
            case 
                when trim(DAT_INSERCAO_CREDITO) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else try_cast(to_timestamp(trim(DAT_INSERCAO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_INSERCAO_CREDITO,
            case 
                when trim(HOR_INSERCAO_CREDITO) in ('null', 'NULL', '', '-3', '-2', '-1') then null
                else
                    cast(substr(lpad(trim(HOR_INSERCAO_CREDITO), 6, '0'), 1, 2) as int) * 3600 +
                    cast(substr(lpad(trim(HOR_INSERCAO_CREDITO), 6, '0'), 3, 2) as int) * 60 +
                    cast(substr(lpad(trim(HOR_INSERCAO_CREDITO), 6, '0'), 5, 2) as int)
            end as HOR_INSERCAO_CREDITO,
            try_cast(DW_NUM_CLIENTE as BIGINT) as DW_NUM_CLIENTE,
            try_cast(COD_TECNOLOGIA_DW as STRING) as COD_TECNOLOGIA_DW,
            try_cast(COD_CANAL_AQUISICAO as INT) as COD_CANAL_AQUISICAO,
            try_cast(COD_TIPO_CREDITO as STRING) as COD_TIPO_CREDITO,
            try_cast(COD_PROMOCAO as INT) as COD_PROMOCAO,
            try_cast(VAL_CREDITO_INSERIDO as DECIMAL(10,2)) as VAL_CREDITO_INSERIDO,
            try_cast(VAL_BONUS as DECIMAL(10,2)) as VAL_BONUS,
            try_cast(VAL_REAL as DECIMAL(10,2)) as VAL_REAL,
            try_cast(COD_PLATAFORMA_ATU as STRING) as COD_PLATAFORMA_ATU,
            try_cast(COD_STATUS_PLATAFORMA as STRING) as COD_STATUS_PLATAFORMA,
            try_cast(IND_METODO_PAGAMENTO as STRING) as IND_METODO_PAGAMENTO,
            try_cast(DW_PLANO_TARIFACAO as INT) as DW_PLANO_TARIFACAO,
            try_cast(DW_TIPO_RECARGA as INT) as DW_TIPO_RECARGA,
            try_cast(DW_TIPO_INSERCAO as INT) as DW_TIPO_INSERCAO,
            try_cast(DW_FORMA_PAGAMENTO as INT) as DW_FORMA_PAGAMENTO,
            try_cast(DW_INSTITUICAO as INT) as DW_INSTITUICAO,
            try_cast(COD_GRUPO_CARTAO as STRING) as COD_GRUPO_CARTAO,
            try_cast(DSC_GRUPO_CARTAO_WPP as STRING) as DSC_GRUPO_CARTAO_WPP,
            try_cast(FLAG_SOS as INT) as FLAG_SOS,
            try_cast(VALOR_SOS as DECIMAL(10,2)) as VALOR_SOS,
            {pdthproc} as DATPROC

        from
            raw_00_com_safra
            
    """.format(pdthproc=dthproc))

lake.createOrReplaceTempView("lake")
lake.count()  

25/12/31 09:18:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


100213651

In [13]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- DW_NUM_NTC: long (nullable = true)
 |-- DAT_INSERCAO_CREDITO: timestamp (nullable = true)
 |-- HOR_INSERCAO_CREDITO: integer (nullable = true)
 |-- DW_NUM_CLIENTE: long (nullable = true)
 |-- COD_TECNOLOGIA_DW: string (nullable = true)
 |-- COD_CANAL_AQUISICAO: integer (nullable = true)
 |-- COD_TIPO_CREDITO: string (nullable = true)
 |-- COD_PROMOCAO: integer (nullable = true)
 |-- VAL_CREDITO_INSERIDO: decimal(10,2) (nullable = true)
 |-- VAL_BONUS: decimal(10,2) (nullable = true)
 |-- VAL_REAL: decimal(10,2) (nullable = true)
 |-- COD_PLATAFORMA_ATU: string (nullable = true)
 |-- COD_STATUS_PLATAFORMA: string (nullable = true)
 |-- IND_METODO_PAGAMENTO: string (nullable = true)
 |-- DW_PLANO_TARIFACAO: integer (nullable = true)
 |-- DW_TIPO_RECARGA: integer (nullable = true)
 |-- DW_TIPO_INSERCAO: integer (nullable = true)
 |-- DW_FORMA_PAGAMENTO: integer (nullable = true)
 |-- DW_INSTITUICAO: int

In [14]:
lake.show(5)

+-----------+------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF| SAFRA|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|       DATPROC|
+-----------+------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+----------

In [15]:
lake_exemplo_dedup = spark.sql("""
    SELECT *
    FROM lake
    WHERE NUM_CPF = "79TUUYWXYWU" AND DW_NUM_NTC = 578905048 AND DAT_INSERCAO_CREDITO = "2023-10-06 00:00:00"
""")

lake_exemplo_dedup.show()

+-----------+------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+--------------------+---------+--------+------------------+---------------------+--------------------+------------------+---------------+----------------+------------------+--------------+----------------+--------------------+--------+---------+--------------+
|    NUM_CPF| SAFRA|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_CANAL_AQUISICAO|COD_TIPO_CREDITO|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAL|COD_PLATAFORMA_ATU|COD_STATUS_PLATAFORMA|IND_METODO_PAGAMENTO|DW_PLANO_TARIFACAO|DW_TIPO_RECARGA|DW_TIPO_INSERCAO|DW_FORMA_PAGAMENTO|DW_INSTITUICAO|COD_GRUPO_CARTAO|DSC_GRUPO_CARTAO_WPP|FLAG_SOS|VALOR_SOS|       DATPROC|
+-----------+------+----------+--------------------+--------------------+--------------+-----------------+-------------------+----------------+------------+----------

In [16]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = lake.dropDuplicates()

lake_dedup.count()


99938358

In [18]:
lake_dedup.createOrReplaceTempView("lake_dedup")

In [19]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT DW_NUM_NTC) as num_tel_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(truncate=False)

+------+------------+-------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|num_tel_distintos|
+------+------------+-------------+-----------------+
|202310|4360883     |1429755      |1759695          |
|202311|4307758     |1448184      |1783264          |
|202312|4603573     |1511629      |1875533          |
|202401|4391992     |1508640      |1866742          |
|202402|4438121     |1513604      |1879546          |
|202403|4973314     |1580382      |1982465          |
|202404|4877590     |1599384      |2011623          |
|202405|5081832     |1650460      |2090755          |
|202406|5134625     |1708324      |2172745          |
|202407|5533341     |1780332      |2292069          |
|202408|5884685     |1821869      |2364586          |
|202409|5831395     |1838158      |2378768          |
|202410|6829005     |2025433      |2660785          |
|202411|7167092     |2199993      |2947022          |
|202412|7283303     |2306817      |3117906          |
|202501|6661177     |2297663

In [20]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_recarga/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.DAT_INSERCAO_CREDITO = s.DAT_INSERCAO_CREDITO
            AND t.DW_NUM_NTC = s.DW_NUM_NTC
            AND t.HOR_INSERCAO_CREDITO = s.HOR_INSERCAO_CREDITO
            AND t.VAL_CREDITO_INSERIDO = s.VAL_CREDITO_INSERIDO
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


Tabela silver não existe. Criando...


In [21]:
spark.stop()